In [ ]:
!pip install -U transformers datasets accelerate sentencepiece evaluate --quiet

# imports

In [ ]:
import os
import gc
import random
import shutil
import time
from dataclasses import dataclass
from typing import Dict, List, Optional
import evaluate
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR
from torch.optim.optimizer import Optimizer
import math

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorForSeq2Seq,
)

try:
    from safetensors.torch import load_file as safe_load, save_file as safe_save
except Exception:
    safe_load, safe_save = None, None

from transformers.utils import logging
logging.set_verbosity_error()   # hides WARNING/INFO messages (incl. load reports)


In [ ]:
def cosine_lr(initial_lr: float,
              round_idx: int,
              total_rounds: int,
              eta_min: float = 0.0) -> float:
    round_idx = min(round_idx, total_rounds)
    cosine_decay = 0.5 * (1 + math.cos(math.pi * round_idx / total_rounds))
    return eta_min + (initial_lr - eta_min) * cosine_decay

In [ ]:
# from huggingface_hub import login
# login(os.environ["HF_TOKEN"])  # optional; public models/datasets need no token

# Constants

In [ ]:
seed = 342365
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



MODEL_NAME = "google/electra-small-discriminator"
Main_path = "./main_model"


OUTPUT_DIR = "electra"
MAIN_PATH_MODEL = Main_path
global_adapter_dir = Main_path
ADAPTER_BIN = "adapter_model.bin"
ADAPTER_SAFE = "adapter_model.safetensors"

EPOCHS = 1
LR = 3.7e-4
rounds = 100
NUM_LABELS = 2
MAX_LENGTH = 96
BATCH_SIZE = 512
num_of_client = 100
num_of_Part = 10

# model Import

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=NUM_LABELS)
model.to(device)
model.save_pretrained(MAIN_PATH_MODEL)
model.save_pretrained("./pre_model")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_params)

# Data

In [ ]:
import pandas as pd
test_url='https://huggingface.co/datasets/stanfordnlp/sentiment140/resolve/refs%2Fconvert%2Fparquet/sentiment140/test/0000.parquet'
!wget {test_url}

train_url='https://huggingface.co/datasets/stanfordnlp/sentiment140/resolve/refs%2Fconvert%2Fparquet/sentiment140/train/0000.parquet'
!wget {train_url}

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [ ]:
df_test = pd.read_parquet('0000.parquet')
df_train = pd.read_parquet('0000.parquet.1')
df_train.info()
df_test  = df_test[df_test["sentiment"] != 2]
df_train["label"] = (df_train["sentiment"] == 4).astype("int8")
df_test["label"]  = (df_test["sentiment"] == 4).astype("int8")

X = df_train.drop(columns=["label"])
y = df_train["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.005,
    stratify=y,
    random_state=seed
)

train_df = X_train.copy()
train_df["label"] = y_train

test_df = X_test.copy()
test_df["label"] = y_test
print(len(train_df),len(test_df))

In [ ]:
label_counts = df_test["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(df_test)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
label_counts = df_train["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(df_train)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
label_counts = test_df["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(test_df)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
hf_valid = Dataset.from_pandas(test_df, preserve_index=False)
hf_test  = Dataset.from_pandas(df_test, preserve_index=False)

def preprocess_fn(examples):
    encodings = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH,
    )
    encodings["labels"] = examples["label"]
    return encodings

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
cols = ["input_ids", "attention_mask", "labels"]

tokenized_valid = hf_valid.map(preprocess_fn, batched=True, remove_columns=hf_valid.column_names)
tokenized_test  = hf_test.map(preprocess_fn, batched=True, remove_columns=hf_test.column_names)

tokenized_valid.set_format("torch", columns=cols)
tokenized_test.set_format("torch", columns=cols)

valid_loader = DataLoader(tokenized_valid, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader  = DataLoader(tokenized_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)


# functions

In [ ]:
s = np.array([np.random.choice(np.arange(num_of_client), size=10, replace=False) for _ in range(200)])
def get_trained_client(setting,numberOfClient,numberParticpate):

    clients = list(range(numberOfClient))

    if setting == 1:
        return clients

    elif setting == 2:
        return  random.sample(clients, numberParticpate)

    elif setting == 3:
        count = random.randint(1, len(clients))
        return random.sample(clients, count)
    elif setting == 4:
        senario = s
        return senario[numberParticpate]
    else:
        raise ValueError("no existing setting that can be used")

In [ ]:
@torch.no_grad()
def evaluate_valid_loss(model, loader: DataLoader, device):
    """
    Computes mean token-level negative log-likelihood (NLL)
    over the validation set, without using HuggingFace's
    built-in loss.

    Returns:
        valid_loss (float): mean NLL over all non-masked tokens
    """
    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        # logits: [batch_size, seq_len, vocab_size]
        logits = outputs.logits

        # Compute log-probabilities
        log_probs = F.log_softmax(logits, dim=-1)

        # Flatten tensors
        labels_flat = labels.view(-1)
        log_probs_flat = log_probs.view(-1, log_probs.size(-1))

        # Mask out ignored labels (-100)
        valid_mask = labels_flat != -100
        valid_labels = labels_flat[valid_mask]

        # Gather log-probabilities of the true labels
        token_log_probs = log_probs_flat[valid_mask, valid_labels]

        # Accumulate NLL
        total_nll += -token_log_probs.sum()
        total_tokens += valid_labels.numel()

    # Mean NLL over entire validation set
    valid_loss = (total_nll / total_tokens).item()

    return valid_loss

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def evaluate_model(model, device, test_loader):
    """
    Hugging Face–compatible evaluation function

    Args:
        model: Hugging Face model (e.g. AutoModelForSequenceClassification)
        device: torch.device
        test_loader: DataLoader (HF-style batches)
    Returns:
        dict with validation_loss, accuracy, f1
    """
    model.to(device)
    model.eval()

    all_logits = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in test_loader:
            # HF batches are dicts
            inputs = {k: v.to(device) for k, v in batch.items()}

            # Forward pass (HF computes loss internally)
            outputs = model(**inputs)

            loss = outputs.loss
            logits = outputs.logits
            labels = inputs["labels"]

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / total_samples

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    preds = np.argmax(all_logits, axis=1)

    metrics = {
        "validation_loss": avg_loss,
        "accuracy": accuracy.compute(
            predictions=preds,
            references=all_labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds,
            references=all_labels
        )["f1"]
    }

    return metrics

In [ ]:
def _int_counts_from_simplex(prob, total, rng):
    """
    Convert simplex vector prob into integer counts that sum to total
    using floor + remainder distribution.
    """
    raw = prob * total
    counts = np.floor(raw).astype(int)
    rem = total - counts.sum()
    if rem > 0:
        frac = raw - counts
        # deterministic tie-breaking with rng noise
        order = np.argsort(-frac + 1e-12 * rng.random(len(frac)))
        counts[order[:rem]] += 1
    return counts

def dirichlet_partition_both(
    labels,
    num_clients: int = 10,
    alpha: float = 0.5,
    beta: float | None = None,
    prior: str = "uniform",      # you asked for uniform
    min_client_size: int = 1,
    seed: int = 42,
):

    rng = np.random.default_rng(seed)
    y = np.asarray(labels)
    N = len(y)

    # Map labels to 0..L-1
    classes = np.unique(y)
    L = len(classes)
    to_int = {c: i for i, c in enumerate(classes)}
    y_int = np.array([to_int[v] for v in y], dtype=int)

    # Prior p over labels (uniform per your requirement)
    if prior == "uniform":
        p = np.ones(L) / L
    elif prior == "empirical":
        cnt = np.bincount(y_int, minlength=L)
        p = cnt / cnt.sum()
    else:
        raise ValueError("prior must be 'uniform' or 'empirical'.")

    # ----- quantity sizes -----
    if beta is None:
        # balanced: same or very similar number of examples
        base = N // num_clients
        rem = N % num_clients
        sizes = [base + (1 if i < rem else 0) for i in range(num_clients)]
        z = None
    else:

        z = rng.dirichlet(np.ones(num_clients) * beta)
        sizes = _int_counts_from_simplex(z, N, rng).tolist()

        # enforce min_client_size if requested
        if min_client_size > 1:
            if min_client_size * num_clients > N:
                raise ValueError("min_client_size too large for total dataset size.")
            # raise small clients up to min by taking from larger ones
            deficit = sum(max(0, min_client_size - s) for s in sizes)
            if deficit > 0:
                order = np.argsort(-np.array(sizes))
                for i in order:
                    if deficit <= 0:
                        break
                    spare = sizes[i] - min_client_size
                    if spare > 0:
                        take = min(spare, deficit)
                        sizes[i] -= take
                        deficit -= take
                for j in range(num_clients):
                    need = max(0, min_client_size - sizes[j])
                    sizes[j] += need

            # re-fix sum exactly to N
            diff = N - sum(sizes)
            if diff > 0:
                for _ in range(diff):
                    sizes[rng.integers(num_clients)] += 1
            elif diff < 0:
                diff = -diff
                for _ in range(diff):
                    candidates = [i for i, s in enumerate(sizes) if s > min_client_size]
                    i = rng.choice(candidates)
                    sizes[i] -= 1

    # ----- label distributions -----

    conc = np.clip(alpha * p, 1e-12, None)  # ensure positive concentration
    q = rng.dirichlet(conc, size=num_clients)

    # Pools of indices per label (for sampling without replacement)
    pools = []
    for k in range(L):
        idxs = np.where(y_int == k)[0]
        rng.shuffle(idxs)
        pools.append(list(idxs))

    # Allocate without replacement according to q_j
    client_indices = []
    for j in range(num_clients):
        m = sizes[j]
        desired = rng.multinomial(m, q[j])

        assigned = []
        # take desired from each pool
        for k in range(L):
            take = min(desired[k], len(pools[k]))
            if take > 0:
                assigned.extend(pools[k][:take])
                pools[k] = pools[k][take:]

        while len(assigned) < m:
            avail = np.array([len(pools[k]) for k in range(L)])
            if avail.sum() == 0:
                break
            mask = (avail > 0).astype(float)
            probs = q[j] * mask
            probs = probs / probs.sum()
            k = rng.choice(L, p=probs)
            assigned.append(pools[k].pop())  # remove => without replacement

        rng.shuffle(assigned)
        client_indices.append(assigned)

    return client_indices, q, sizes, classes, z

In [ ]:

def make_client_shards(df, client_indices):
    """
    df: pandas.DataFrame
    client_indices: list[list[int]]  (row indices per client)
    returns: list[pandas.DataFrame]
    """
    return [df.iloc[idxs].reset_index(drop=True) for idxs in client_indices]


In [ ]:
client_idxs, q, sizes, classes, z = dirichlet_partition_both(
    labels=train_df["label"],
    num_clients=num_of_client,
    alpha=0.5,          # strong label non-IID
    beta=None,          # balanced client sizes
    prior="uniform",
    seed=seed
)
print("sizes:", sizes, "min/max:", min(sizes), max(sizes))

In [ ]:
all_ids = np.concatenate([np.array(x) for x in client_idxs])
print("Assigned:", len(all_ids), "Unique:", len(np.unique(all_ids)))

In [ ]:
client_shards = make_client_shards(train_df, client_idxs)

print("Client sizes:", [len(c) for c in client_shards])
print("Min/Max:", min(len(c) for c in client_shards), max(len(c) for c in client_shards))

print("Example client[0]:")
print(client_shards[0].iloc[0])


In [ ]:
train_proc = [ Dataset.from_pandas(train_ds, preserve_index=False) for train_ds in client_shards]
train_proc = [ train_ds.map(preprocess_fn, batched=True, remove_columns=train_ds.column_names) for train_ds in train_proc]
for i in train_proc:
        i.set_format(type="torch", columns=cols)

In [ ]:
train_loader = [DataLoader(i, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator) for i in train_proc]

# model realated function

In [ ]:
def base_model_loader(adapter_dir: str):
    model = AutoModelForSequenceClassification.from_pretrained(adapter_dir,num_labels=2)
    return model

In [ ]:
def _move_optimizer_state_to_device(opt_state, device):
    for state in opt_state.get("state", {}).values():
        for k, v in list(state.items()):
            if torch.is_tensor(v):
                state[k] = v.to(device, non_blocking=True)
            elif isinstance(v, (list, tuple)):
                state[k] = [t.to(device, non_blocking=True) if torch.is_tensor(t) else t for t in v]
            elif isinstance(v, dict):
                for kk, vv in list(v.items()):
                    if torch.is_tensor(vv):
                        v[kk] = vv.to(device, non_blocking=True)

In [ ]:
def save_full_model_state_dict(state_dict, out_dir: str):
    """
    Save FULL model weights (HF-compatible).
    Deletes existing directory to avoid stale files.
    """
    if os.path.exists(out_dir):
        print("removed")
        shutil.rmtree(out_dir)

    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, "pytorch_model.bin")
    torch.save(state_dict, out_path)


def load_full_model_state_dict(model_dir: str, map_location="cpu"):
    """
    Load FULL model weights from HF-style directory.
    """
    bin_path = os.path.join(model_dir, "pytorch_model.bin")
    safe_path = os.path.join(model_dir, "model.safetensors")

    if os.path.exists(bin_path):
        return torch.load(bin_path, map_location=map_location)
    elif safe_load is not None and os.path.exists(safe_path):
        return safe_load(safe_path)
    else:
        raise FileNotFoundError(f"No model weights found in {model_dir}")


# ============================================================
# 2) FEDAVG OVER FULL MODELS (CPU SAFE)
# ============================================================

@torch.no_grad()
def average_full_model_states(model_dirs, weights=None):
    """
    FedAvg over FULL model parameters on CPU.
    Safely skips non-floating tensors.
    """
    n = len(model_dirs)
    if n == 0:
        raise ValueError("No model dirs provided.")

    if weights is None:
        weights = [1.0 / n] * n
    else:
        s = sum(weights)
        weights = [w / s for w in weights]

    avg = load_full_model_state_dict(model_dirs[0], map_location="cpu")

    for k, v in avg.items():
        if torch.is_floating_point(v):
            avg[k] = v.float() * weights[0]

    for w, d in zip(weights[1:], model_dirs[1:]):
        sd = load_full_model_state_dict(d, map_location="cpu")
        for k in avg:
            if torch.is_floating_point(avg[k]):
                avg[k].add_(sd[k].float(), alpha=w)

    return avg


# ============================================================
# 4) AGGREGATION CONTROLLER (FULL MODELS ONLY)
# ============================================================
def add_parameters(model, edge, ratio):
  for server_param, edge_param in zip(model.parameters(), edge.parameters()):
    server_param.data += edge_param.data.clone() * ratio

def aggregate_adapters(
    selected_model_dirs,
    algorithm: str,
    global_model_dir: str,
    train_loaders=None,
    client_ids=None,
    wagg_set=None,
    base_model_loader=None,
    device="cuda",
):
    """
    Supported algorithms (FULL MODEL):
      - Sophia   (uniform FedAvg)
      - FedAvg   (data-size weighted)
      - wagg     (NLL-weighted)
    """
    model = base_model_loader(global_model_dir)
    # ============= SOPHIA =================
    if algorithm.lower() == "sophia":
      for p in model.parameters():
          p.data.zero_()
      # equal‐weight average over selected edges
      for edge in selected_model_dirs:
          cloneModel = base_model_loader(edge)
          add_parameters(model,cloneModel, 1 / len(selected_model_dirs))
          del cloneModel                # Remove the variable reference
          torch.cuda.empty_cache()

      model.save_pretrained(global_model_dir)
      del model                # Remove the variable reference
      torch.cuda.empty_cache()

      return

    # ============= FEDAVG =================
    if algorithm.lower() == "fedavg":
      for p in model.parameters():
          p.data.zero_()
      # equal‐weight average over selected edges
      for i,edge in enumerate(selected_model_dirs):
          cloneModel = base_model_loader(edge)
          add_parameters(model,cloneModel, 1 / len(train_loaders[i]))
          del cloneModel                # Remove the variable reference
          torch.cuda.empty_cache()

      model.save_pretrained(global_model_dir)
      del model                # Remove the variable reference
      torch.cuda.empty_cache()
      return

    # ============= WAGG ===================
    if algorithm.lower() == "wagg":

      assert wagg_set is not None
      assert base_model_loader is not None

      selected_model_dirs.append(global_model_dir)

      for p in model.parameters():
        p.data.zero_()

      scores = []
      weights = []
      for d in selected_model_dirs:
        cloneModel = base_model_loader(d).to(device)
        loss = evaluate_valid_loss(cloneModel, wagg_set, device)
        scores.append(1.0 / loss)

        del cloneModel
        torch.cuda.empty_cache()
        gc.collect()

        s = sum(scores)
        weights = [x / s for x in scores] if s > 0 else [1.0 / len(scores)] * len(scores)

      for i,edge in enumerate(selected_model_dirs):
        cloneModel = base_model_loader(edge)
        add_parameters(model,cloneModel,weights[i])
        del cloneModel                # Remove the variable reference
        torch.cuda.empty_cache()

      model.save_pretrained(global_model_dir)
      del model                # Remove the variable reference
      torch.cuda.empty_cache()
      return

    # ============= FEDLAW ==================
    if algorithm.lower() == "fedlaw":
        raise RuntimeError(
            "FedLAW is adapter-only. "
            "Scaling and fusing FULL model parameters is unstable "
            "(LayerNorm / embeddings break)."
        )

    raise ValueError(f"Unknown algorithm: {algorithm}")

In [ ]:
def get_client_state_paths(edge: int, base_state_dir="client_state"):
    state_dir = os.path.join(base_state_dir, f"edge_{edge}")
    os.makedirs(state_dir, exist_ok=True)
    opt_path = os.path.join(state_dir, "optimizer.pt")
    return state_dir, opt_path

def get_round_adapter_dir(round_id: int, edge: int, cache_root="round_cache"):
    d = os.path.join(cache_root, f"round_{round_id}", f"edge_{edge}")
    os.makedirs(d, exist_ok=True)
    return d

# train loop

In [ ]:
print("main")
test_acc = evaluate_model(model, device,test_loader)
print(f" global_loss={test_acc["validation_loss"]:.4f} global_acc = {test_acc["accuracy"]:.4f} f1={test_acc["f1"]:.4f}")


In [ ]:
def train_firstorder(
    base_model_loader,          # function(adapter_dir) -> model (base + adapter loaded)
    global_adapter_dir: str,    # current global adapter checkpoint directory
    train_loaders,
    test_loader,
    valid_loaders=None,         # if you use it elsewhere
    algorithm="Sophia",
    grad_clip=1.0,
    lr=3e-5,
    epochs=1,
    rounds=10,
    num_clients=10,
    clients_per_round=4,
    seed=42,
    device="cuda",
    k = 10):

    results = []

    for globalRound in range(100):
        print(f"\n##################  Round {globalRound}  ########################")

        chosen = get_trained_client(4, num_clients, globalRound)

        selected_adapter_dirs = []
        selected_client_ids = []

        for edge in range(num_clients):
            if edge not in chosen:
                continue

            print(f"\n################## edge {edge} ########################")

            # ephemeral adapter output for this round (can delete later)
            out_adapter_dir = get_round_adapter_dir(globalRound, edge)
            _ , opt_path = get_client_state_paths(edge)
            # ---- load model (base + current global adapter) to GPU ----
            model = base_model_loader(adapter_dir=global_adapter_dir).to(device)
            model.train()
            new_lr = cosine_lr(initial_lr=lr,round_idx=globalRound,total_rounds=rounds,eta_min=5e-7)
            params = [p for p in model.parameters() if p.requires_grad]
            #print(model.print_trainable_parameters())
            optimizer = torch.optim.Adam(
                params,
                lr=new_lr,
            )
            if os.path.exists(opt_path):
                opt_state = torch.load(opt_path, map_location="cpu")
                _move_optimizer_state_to_device(opt_state, device)
                optimizer.load_state_dict(opt_state)
                print(f"Loaded optimizer state for edge {edge} from {opt_path}")

            global_steps = 0
            # ---- local training ----
            for epoch in range(1, epochs + 1):
                running_loss = 0.0

                for step, batch in enumerate(train_loaders[edge], start=1):
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)

                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )
                    loss = outputs.loss
                    loss.backward()

                    optimizer.step()
                    running_loss += float(loss.detach().cpu())
                    global_steps = global_steps + 1
                    # ---- Hessian update (cheap-ish) ----
                    # IMPORTANT: no graph should be kept alive from above.

                avg_train_loss = running_loss / max(1, len(train_loaders[edge]))
                val_acc= evaluate_model(model, device, test_loader)  # you can swap to a smaller client-val loader
                print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}  val_loss={val_acc["validation_loss"]:.4f} acc={val_acc["accuracy"]:.4f} f1={val_acc["f1"]:.4f}")

            # ---- save ONLY adapter + optimizer state ----
            model.save_pretrained(out_adapter_dir)
            torch.save(optimizer.state_dict(), opt_path)

            selected_adapter_dirs.append(out_adapter_dir)
            selected_client_ids.append(edge)

            # ---- free GPU memory (your requirement) ----
            del model
            del optimizer
            torch.cuda.empty_cache()
            gc.collect()

        # ---- server aggregation (adapter-only, CPU) ----
        aggregate_adapters(
            selected_adapter_dirs,
            algorithm=algorithm,
            global_model_dir=global_adapter_dir,  # overwrite/update
            train_loaders=train_loaders,
            client_ids=selected_client_ids,
            wagg_set=valid_loaders if valid_loaders is not None else None,
            base_model_loader=base_model_loader,
            device=device,
        )

        # ---- evaluate global model ----
        global_model = base_model_loader(adapter_dir=global_adapter_dir).to(device)
        test_acc = evaluate_model(global_model, device,test_loader)
        #global_model.save_pretrained(global_adapter_dir)
        results.append(test_acc)
        print(f"Round {globalRound}: global_loss={test_acc["validation_loss"]:.4f} global_acc = {test_acc["accuracy"]:.4f} f1={val_acc["f1"]:.4f}")

        del global_model
        torch.cuda.empty_cache()
        gc.collect()

        # ---- delete per-round cache to save disk (optional) ----
        round_dir = os.path.join("round_cache", f"round_{globalRound}")
        if os.path.exists(round_dir) and not(globalRound == rounds-1):
            shutil.rmtree(round_dir, ignore_errors=True)

    return results

# half wagg

In [ ]:
#indx =  [7946, 5323, 183, 3179, 5678, 2901, 2701, 4034, 6942, 6983, 560, 7245, 820, 7452, 4564, 9, 3290, 3086, 7652, 6744, 5393, 3666, 904, 4940, 42, 319, 2069, 55, 3387, 6980, 382, 3401, 1001, 2354, 3147, 1345, 1668, 7323, 2314, 5361, 494, 3525, 1424, 212, 421, 461, 516, 543, 631, 839, 1107, 1263, 1399, 1609, 1629, 2304, 2376, 2708, 3291, 3320, 3540, 3617, 3694, 3697, 4011, 4091, 4350, 4676, 4807, 5028, 5288, 5399, 5488, 5593, 5600, 6088, 6171, 6316, 6322, 6450, 6526, 7087, 7503, 7521, 7867, 7968]
indx =  [475, 755, 895, 927, 1089, 1147, 1951, 1968, 2006, 2296, 2752, 3056, 3120, 3157, 3406, 3469, 3515, 3795, 3860, 4710, 5475, 5720, 5862, 5957, 6002, 6117, 6296, 6623, 6624, 6996, 7137, 7274, 7451, 7523, 7542, 7953, 6954, 7882, 5381, 6509, 4709, 4901, 1690, 1340, 7321, 7760, 503, 4087, 5533, 1538, 1195, 692, 977, 7958, 2095, 4642, 4430, 730, 3075, 294, 590, 7461, 4987, 7186, 3158, 3069, 2519, 762, 2559, 3114, 922, 7095]
print(len(indx))

In [ ]:
h2 = hf_valid.select(indx)
h1_token = h2.map(preprocess_fn, batched=True, remove_columns=h2.column_names)
h1_token.set_format(type="torch", columns=cols)

valid_loader_1 = DataLoader(h1_token, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)


In [ ]:
#del model
torch.cuda.empty_cache()
gc.collect()
start_time = time.time()  # Record start time
results = train_firstorder(
    base_model_loader,          # function(adapter_dir) -> model (base + adapter loaded)
    global_adapter_dir = Main_path,    # current global adapter checkpoint directory
    train_loaders = train_loader,
    test_loader = test_loader,
    valid_loaders= valid_loader_1 ,         # if you use it elsewhere
    algorithm="wagg",
    grad_clip=1.0,
    lr=LR,
    epochs=EPOCHS,
    rounds=rounds,
    num_clients=num_of_client,
    clients_per_round=num_of_Part,
    seed=seed,
    device=device)         # Call the function
end_time = time.time()  # Record end time

execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

In [ ]:
for i in range(len(results)):
    print(f"round {i} = {results[i]}")